## This program creates input for the metrics aggregator
* Fuses prs and comments
* Filters repos by stars
* Fixes fields and date values

In [176]:
import pandas as pd
import json

In [177]:
desired_time_format = "%m/%d/%y, %I:%M:%S %p"

In [179]:
# load in all required data
all_repos = pd.read_parquet('./data/all_repository.parquet')

ai_prs = pd.read_parquet('./data/all_pull_request.parquet')
human_prs = pd.read_parquet('./data/human_pull_request.parquet')

ai_comments = pd.read_parquet('./data/pr_comments.parquet')
with open("./data/human_comments.json", encoding = "utf-8") as file:
    human_comments = json.load(file)

In [180]:
ai_prs = ai_prs.drop(columns='repo_id')
print(f"Ai entries: {len(ai_prs)}")
print(f"Human entries: {len(human_prs)}")

Ai entries: 932791
Human entries: 6618


In [181]:
human_prs.head()

,id,number,title,user,user_id,state,created_at,closed_at,merged_at,repo_url,html_url,body,agent
0,2336888723,85268,feat(aci): add automations index page,ameliahsu,55610339,closed,2025-02-14T19:04:59Z,2025-02-18T22:42:20Z,2025-02-18T22:42:19Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85268,https://sentry-j41gpomr5.sentry.dev/automation...,Human
1,2447123365,89131,ref(insights): Make use of `<FeatureBadge>` fo...,ryan953,187460,closed,2025-04-08T23:29:50Z,2025-04-09T15:56:55Z,2025-04-09T15:56:54Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/89131,Using the premade component reduces an import ...,Human
2,2438086945,88748,:bug: fix: update how we fetch workflow_id and...,iamrajjoshi,33237075,closed,2025-04-03T21:36:59Z,2025-04-04T15:10:57Z,2025-04-04T15:10:57Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/88748,i realized i made a mistake for how i fetch th...,Human
3,2265431531,83085,fix(org-stats): Require project membership,ArthurKnaus,7033940,closed,2025-01-08T07:47:13Z,2025-01-08T08:49:40Z,2025-01-08T08:49:40Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/83085,### Problem\r\n\r\nIf the user is not member o...,Human
4,2332333882,85102,ref(consumers): Rename parallel -> batched-par...,evanpurkhiser,1421724,closed,2025-02-12T21:24:17Z,2025-02-12T22:20:33Z,2025-02-12T22:20:33Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85102,Both crons and uptime consumers have a paralle...,Human


In [182]:
print(ai_prs["agent"].unique())
print(human_prs["agent"].unique())

['Claude_Code' 'Copilot' 'OpenAI_Codex' 'Cursor' 'Devin']
['Human']


In [183]:
# get data for popular repos by filtering star count
print(f"All repos: {len(all_repos)}")

pop_repos = all_repos[all_repos["stars"] >= 500]
print(f"Popular repos: {len(pop_repos)}")

All repos: 116211
Popular repos: 1496


In [184]:
# update prs to only those in popular repositories and closed
ai_prs = ai_prs[ai_prs["repo_url"].isin(pop_repos["url"])]
ai_prs = ai_prs[~ai_prs["closed_at"].isna()]

human_prs = human_prs[human_prs["repo_url"].isin(pop_repos["url"])]
human_prs = human_prs[~human_prs["closed_at"].isna()]

print(f"Ai final entries: {len(ai_prs)}")
print(f"Human final entries: {len(human_prs)}")

Ai final entries: 11105
Human final entries: 6129


In [185]:
human_prs["repo_url"].nunique()

773

In [186]:
# reconfigure some elements of the data

#ai data
ai_prs = ai_prs.rename(columns={'user_id': 'userid'})
ai_prs['userid'] = ai_prs['userid'].apply(str)

ai_comments = ai_comments.rename(columns={'user_id': 'userid'})
ai_comments['userid'] = ai_comments['userid'].apply(str)

def change_dates(column):
    return pd.to_datetime(column, format = "mixed").dt.strftime(desired_time_format)

ai_prs['created_at'] = change_dates(ai_prs['created_at'])
ai_prs['closed_at'] = change_dates(ai_prs['closed_at'])
ai_prs['merged_at'] = change_dates(ai_prs['merged_at'])
ai_comments['created_at'] = change_dates(ai_comments['created_at'])

In [268]:
#human data
human_prs = human_prs.rename(columns={'user_id': 'userid'})
human_prs['userid'] = human_prs['userid'].apply(str)

#create a dataframe with human comments
comments_list = []
current_comment = []
html_url = lambda repo, pr: "https://github.com/" + str(repo) + "/pull/" + str(pr) 
usertype = lambda username: "Bot" if ("[bot]" in username or "[Bot]" in username) else "User"
content = lambda x: [x["userlogin"], x["userid"], usertype(x["userlogin"]), x["created_at"], x["body"]]

for repo, prs in human_comments.items():
    for pr, comments in prs.items():
        comments = comments["comments"]
        if comments:
            for comment in comments.values():
                current = [html_url(repo, pr)] + [repo, pr] + content(comment)
                comments_list = comments_list + [current]
                
cols = ["pr_url"] + list(ai_comments.keys())
human_coms = pd.DataFrame(comments_list, columns = cols)

#fix dates
human_prs['created_at'] = change_dates(human_prs['created_at'])
human_prs['closed_at'] = change_dates(human_prs['closed_at'])
human_prs['merged_at'] = change_dates(human_prs['merged_at'])
human_coms['created_at'] = change_dates(human_coms['created_at'])

In [269]:
human_coms.head()

,pr_url,id,pr_id,user,userid,user_type,created_at,body
0,https://github.com/getsentry/sentry/pull/85268,getsentry/sentry,85268,codecov[bot],22429695,Bot,"02/18/25, 10:24:49 PM",## [Bundle](https://app.codecov.io/gh/getsentr...
1,https://github.com/getsentry/sentry/pull/89131,getsentry/sentry,89131,codecov[bot],22429695,Bot,"04/08/25, 11:33:46 PM",## [Codecov](https://app.codecov.io/gh/getsent...
2,https://github.com/getsentry/sentry/pull/89131,getsentry/sentry,89131,gggritso,989898,User,"04/09/25, 01:37:10 PM","I don't know if this is Sentry-wide, but the I..."
3,https://github.com/getsentry/sentry/pull/89131,getsentry/sentry,89131,ryan953,187460,User,"04/09/25, 03:56:50 PM",> I don't know if this is [Sentry-wide](https:...
4,https://github.com/getsentry/sentry/pull/85102,getsentry/sentry,85102,codecov[bot],22429695,Bot,"02/12/25, 09:47:31 PM",## [Codecov](https://app.codecov.io/gh/getsent...


In [236]:
print(len(human_coms))
print(len([name for name in human_coms["user_type"] if name == "Bot"]))

4825
3116


In [258]:
ai_comments.head()

,id,pr_id,user,userid,user_type,created_at,body
0,2927293042,3107321792,coderabbitai[bot],136622811,Bot,"06/01/25, 02:15:35 PM",<!-- This is an auto-generated comment: summar...
1,3090154270,3234660269,Fank,1900106,User,"07/18/25, 05:22:34 PM","claude budget reached, development is on hold ..."
2,2848667986,3037457814,wilsonccccc,6146503,User,"05/03/25, 03:12:37 PM","Hi, thanks for sharing and contributing! Pleas..."
3,2719183506,2915198291,cloudflare-workers-and-pages[bot],73139402,Bot,"03/12/25, 09:34:46 PM",## Deploying instructor-py with &nbsp;<a href=...
4,3013998830,3132410695,razimantv,3823215,User,"06/27/25, 06:16:05 PM",Can you rebase it to the current master? Pleas...


In [270]:
def combine_pr_and_comments(prs, comments):
    pr_num_column = "pr_url" if "pr_url" in comments.keys() else "pr_id"
    pr_key_column = "html_url" if pr_num_column == "pr_url" else "id"
    
    pr_key_comments = {}
    
    #put all the comments into a dictionary by pr_id
    indexes = list(comments.columns)
    for comment in comments.iterrows():
        comment = comment[1]
        
        pr_id = comment[pr_num_column]
        if pr_id in pr_key_comments:
            pr_key_comments[pr_id].append(comment)
        else:
            pr_key_comments[pr_id] = [comment]
    
    #add in new columns for comments and num_comments
    prs["num_comments"] = 0
    prs["comments"] = [{} for _ in range(len(prs))] #change type if structure of comments changes
    
    
    #put prs into a new list and add the comments that correspond to the pr
    new_prs = []
    
    for pr in prs.iterrows():
        pr = pr[1]
        
        pr_comments = pr_key_comments.get(pr[pr_key_column], None)
        
        if pr_comments is not None:
            pr["num_comments"] = len(pr_comments)
            pr["comments"] = {i : comment for i, comment in enumerate(pr_comments)} #create a dict with index:comment pairs
        
        new_prs.append(pr)
         
    #return a new dataframe from the prs with comments
    combined_df = pd.DataFrame(new_prs)
    return combined_df

In [271]:
ai_combined_df = combine_pr_and_comments(ai_prs, ai_comments)
human_combined_df = combine_pr_and_comments(human_prs, human_coms)

In [272]:
print(len(ai_comments))
print(sum(ai_combined_df["num_comments"]))

39122
24559


In [273]:
print(len(human_coms))
print(sum(human_combined_df["num_comments"]))

4825
4341


In [274]:
ai_combined_df.head()

,id,number,title,body,agent,userid,user,state,created_at,closed_at,merged_at,repo_url,html_url,num_comments,comments
7,3264428170,464,🚀 Complete 64-Agent System Implementation,# 🚀 Complete 64-Agent System Implementation\n\...,Claude_Code,2934394,ruvnet,closed,"07/25/25, 09:26:37 PM","07/25/25, 09:29:18 PM",NaN,https://api.github.com/repos/ruvnet/claude-flow,https://github.com/ruvnet/claude-flow/pull/464,0,{}
26,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,108661493,weiliu1031,closed,"07/26/25, 02:59:01 AM","07/29/25, 07:01:20 AM",NaN,https://api.github.com/repos/milvus-io/pymilvus,https://github.com/milvus-io/pymilvus/pull/2911,2,"{0: [3121064306, 3264933329, 'sre-ci-robot', '..."
77,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,7475,MSch,closed,"07/26/25, 01:31:19 PM","07/26/25, 01:37:22 PM","07/26/25, 01:37:22 PM",https://api.github.com/repos/steipete/Peekaboo,https://github.com/steipete/Peekaboo/pull/30,0,{}
192,3234102722,318,chore: Convert hive-mind coordination system t...,## Summary\n\nThis PR converts the AI agent co...,Claude_Code,15803865,lanemc,closed,"07/16/25, 01:00:34 AM","07/17/25, 12:49:29 PM",NaN,https://api.github.com/repos/ruvnet/claude-flow,https://github.com/ruvnet/claude-flow/pull/318,0,{}
238,3214555104,16658,Add function signature breaking change detector,<details><summary>&#x1F6E0 DevTools &#x1F6E0</...,Claude_Code,17039389,harupy,closed,"07/09/25, 05:35:26 AM","07/11/25, 05:13:35 AM","07/11/25, 05:13:35 AM",https://api.github.com/repos/mlflow/mlflow,https://github.com/mlflow/mlflow/pull/16658,3,"{0: [3056782342, 3214555104, 'serena-ruan', '8..."


In [275]:
human_combined_df.head()

,id,number,title,user,userid,state,created_at,closed_at,merged_at,repo_url,html_url,body,agent,num_comments,comments
0,2336888723,85268,feat(aci): add automations index page,ameliahsu,55610339,closed,"02/14/25, 07:04:59 PM","02/18/25, 10:42:20 PM","02/18/25, 10:42:19 PM",https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85268,https://sentry-j41gpomr5.sentry.dev/automation...,Human,1,{0: ['https://github.com/getsentry/sentry/pull...
1,2447123365,89131,ref(insights): Make use of `<FeatureBadge>` fo...,ryan953,187460,closed,"04/08/25, 11:29:50 PM","04/09/25, 03:56:55 PM","04/09/25, 03:56:54 PM",https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/89131,Using the premade component reduces an import ...,Human,3,{0: ['https://github.com/getsentry/sentry/pull...
2,2438086945,88748,:bug: fix: update how we fetch workflow_id and...,iamrajjoshi,33237075,closed,"04/03/25, 09:36:59 PM","04/04/25, 03:10:57 PM","04/04/25, 03:10:57 PM",https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/88748,i realized i made a mistake for how i fetch th...,Human,0,{}
3,2265431531,83085,fix(org-stats): Require project membership,ArthurKnaus,7033940,closed,"01/08/25, 07:47:13 AM","01/08/25, 08:49:40 AM","01/08/25, 08:49:40 AM",https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/83085,### Problem\r\n\r\nIf the user is not member o...,Human,0,{}
4,2332333882,85102,ref(consumers): Rename parallel -> batched-par...,evanpurkhiser,1421724,closed,"02/12/25, 09:24:17 PM","02/12/25, 10:20:33 PM","02/12/25, 10:20:33 PM",https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85102,Both crons and uptime consumers have a paralle...,Human,1,{0: ['https://github.com/getsentry/sentry/pull...


In [276]:
ai_combined_df.reset_index(drop=True, inplace=True) # .to_json was complaining about indexing so this is a fix
human_combined_df.reset_index(drop=True, inplace=True)

In [277]:
ai_combined_df.to_json("ai_combined.json", orient="index", indent=4) #indent is for readability, may increase file size

In [278]:
human_combined_df.to_json("human_combined.json", orient="index", indent = 4)